In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Ad_Click_Through_Rate_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Ad_Click_Through_Rate_Prediction'

In [5]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class DataPreprocessingConfig:
    root_dir: Path

    input_data_path: Path

    output_data_path: Path

    preprocessing_report_path: Path

In [6]:
from src.ad_ctr_prediction.constants import *
from src.ad_ctr_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_preprocessing_config(self) -> DataPreprocessingConfig:

        config = self.config.data_preprocessing

        create_directories(
            [config.root_dir]
        )

        return DataPreprocessingConfig(

            root_dir=Path(config.root_dir),

            input_data_path=Path(config.input_data_path),

            output_data_path=Path(config.output_data_path),

            preprocessing_report_path=Path(config.preprocessing_report_path)
        )    

In [9]:
import json
import numpy as np
import pandas as pd
from src.ad_ctr_prediction import logger

In [10]:
class DataPreprocessing:

    def __init__(self, config: DataPreprocessingConfig):

        self.config = config

        self.data = pd.read_csv(
            self.config.input_data_path
        )

        self.report = {}

    def remove_duplicates(self):

        before_rows = len(self.data)

        self.data.drop_duplicates(inplace=True)

        after_rows = len(self.data)

        removed = (before_rows - after_rows)

        self.report["duplicates_removed"] = removed

        logger.info(
            f"Removed {removed} duplicates"
        )  

    def handle_infinite_values(self):

        numeric_cols = (self.data.select_dtypes(include=np.number).columns)

        total_inf = 0

        for col in numeric_cols:

            count = np.isinf(self.data[col]).sum()

            total_inf += count

        self.data.replace([np.inf, -np.inf], np.nan, inplace=True)

        self.report["infinite_values_replaced"] = int(total_inf)    

    def handle_missing_numerical(self):

        numeric_cols = (
            self.data.select_dtypes(include=np.number).columns)

        for col in numeric_cols:

            self.data[col] = (
                self.data[col].fillna(self.data[col].median())
            )

        logger.info(
            "Numerical missing values handled"
        )      

    def handle_missing_categorical(self):

        cat_cols = (
            self.data.select_dtypes(include=["object"]).columns
        )

        for col in cat_cols:

            self.data[col] = (
                self.data[col].fillna("Unknown")
            )

        logger.info(
            "Categorical missing values handled"
        )    

    def clean_invalid_ctr_values(self):

        ctr_columns = [

            "user_historical_ctr",

            "user_conversion_rate",

            "advertiser_historical_ctr",

            "advertiser_budget_utilization"
        ]

        before_rows = len(
            self.data
        )

        for col in ctr_columns:

            if col in self.data.columns:

                self.data = self.data[
                    (
                        self.data[col] >= 0
                    )
                    &
                    (
                        self.data[col] <= 1
                    )
                ]

        removed_rows = (before_rows - len(self.data))

        self.report["invalid_ctr_rows_removed"] = int(removed_rows)   

        logger.info(
            f"Removed {removed_rows} rows with invalid CTR values"
        ) 

    def process_timestamp(self):

        if ("timestamp" not in self.data.columns):
            return

        self.data["timestamp"] = (
            pd.to_datetime(
                self.data["timestamp"]
            )
        )

        self.data["year"] = (
            self.data["timestamp"].dt.year
        )

        logger.info(
            "Processed timestamp into year"
        )

    def handle_outliers(self):

        numeric_cols = (
            self.data.select_dtypes(include=np.number).columns)

        for col in numeric_cols:

            lower = (
                self.data[col].quantile(0.01)
            )

            upper = (
                self.data[col].quantile(0.99)
            )

            self.data[col] = (
                self.data[col].clip(lower, upper)
                )

        logger.info(
            "Outliers handled"
        )    
    
    def save_data(self):

        self.data.to_csv(
            self.config.output_data_path,
            index=False
        )

        with open(self.config.preprocessing_report_path,"w") as file:
            json.dump(
                self.report,
                file,
                indent=4
            )

        logger.info(
            "Preprocessed data saved"
        )

    def initiate_data_preprocessing(self):

        logger.info(
            "Starting preprocessing"
        )

        self.remove_duplicates()

        self.handle_infinite_values()

        self.handle_missing_numerical()

        self.handle_missing_categorical()

        self.clean_invalid_ctr_values()

        self.process_timestamp()

        self.handle_outliers()

        self.save_data()

        logger.info(
            "Preprocessing completed"
        )

        return (self.config.output_data_path)    

In [11]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_preprocessing_config()
    data_validation = DataPreprocessing(config = data_transformation_config)
    output_data_pth = data_validation.initiate_data_preprocessing()
except Exception as e:
    raise e

[2026-06-19 20:05:33,063: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-19 20:05:33,067: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-19 20:05:33,091: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-06-19 20:05:33,097: INFO: common: created directory at: artifacts]
[2026-06-19 20:05:33,099: INFO: common: created directory at: artifacts/data_preprocessing]
[2026-06-19 20:05:34,175: INFO: 3358652031: Starting preprocessing]
[2026-06-19 20:05:34,536: INFO: 3358652031: Removed 0 duplicates]
[2026-06-19 20:05:35,102: INFO: 3358652031: Numerical missing values handled]
[2026-06-19 20:05:35,314: INFO: 3358652031: Categorical missing values handled]
[2026-06-19 20:05:35,582: INFO: 3358652031: Removed 0 rows with invalid CTR values]
[2026-06-19 20:05:35,779: INFO: 3358652031: Processed timestamp into year]
[2026-06-19 20:05:36,145: INFO: 3358652031: Outliers handled]
[2026-06-19 20:05:42,138: INFO: 3358652031: Preprocessed